In [1]:
#####################################################
# Landsat from USGS
#####################################################

import json
import requests
from getpass import getpass
import sys
import time
import re
import threading
import datetime
import os
import pandas as pd
import geopandas as gpd

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Send http request
def sendRequest(url, data, apiKey = None, exitIfNoResponse = True):
    """
    Send a request to an M2M endpoint and returns the parsed JSON response.

    Parameters:
    endpoint_url (str): The URL of the M2M endpoint
    payload (dict): The payload to be sent with the request

    Returns:
    dict: Parsed JSON response
    """
    
    json_data = json.dumps(data)
    
    if apiKey == None:
        response = requests.post(url, json_data)
    else:
        headers = {'X-Auth-Token': apiKey}              
        response = requests.post(url, json_data, headers = headers)  
    
    try:
      httpStatusCode = response.status_code 
      if response == None:
          print("No output from service")
          if exitIfNoResponse: sys.exit()
          else: return False
      output = json.loads(response.text)
      if output['errorCode'] != None:
          print(output['errorCode'], "- ", output['errorMessage'])
          if exitIfNoResponse: sys.exit()
          else: return False
      if  httpStatusCode == 404:
          print("404 Not Found")
          if exitIfNoResponse: sys.exit()
          else: return False
      elif httpStatusCode == 401: 
          print("401 Unauthorized")
          if exitIfNoResponse: sys.exit()
          else: return False
      elif httpStatusCode == 400:
          print("Error Code", httpStatusCode)
          if exitIfNoResponse: sys.exit()
          else: return False
    except Exception as e: 
          response.close()
          print(e)
          if exitIfNoResponse: sys.exit()
          else: return False
    response.close()
    
    return output['data']

In [3]:
def downloadFile(url):
    sema.acquire()
    try:
        response = requests.get(url, stream=True)
        disposition = response.headers['content-disposition']
        filename = re.findall("filename=(.+)", disposition)[0].strip("\"")
        print(f"    Downloading: {filename}...")
        
        open(os.path.join(data_dir, filename), 'wb').write(response.content)
        sema.release()
    except Exception as e:
        print(f"\nFailed to download from {url}. Will try to re-download.")
        sema.release()
        runDownload(threads, url)

In [4]:
def runDownload(threads, url):
    thread = threading.Thread(target=downloadFile, args=(url,))
    threads.append(thread)
    thread.start()

In [5]:
data_dir = 'data/landsat/Yamal/data'
utils_dir = 'data/landsat/Yamal/utils'
dirs = [ data_dir, utils_dir]

for d in dirs:
        if not os.path.exists(d): 
            try: 
                os.makedirs(d)
                print(f"Directory '{d}' created successfully.") 
            except OSError as e: 
                print(f"Error creating directory '{d}': {e}") 
        else: 
            print(f"Directory '{d}' already exists.") 

Directory 'data/landsat/Yamal/data' already exists.
Directory 'data/landsat/Yamal/utils' already exists.


In [6]:
maxthreads = 5 # Threads count for downloads
sema = threading.Semaphore(value=maxthreads)
label = datetime.datetime.now().strftime("%Y%m%d_%H%M%S") # Customized label using date time
threads = []

In [7]:
def prompt_ERS_login(serviceURL):
    print("Logging in...\n")

    p = ['Enter EROS Registration System (ERS) Username: ', 'Enter ERS Account Token: ']

    # Use requests.post() to make the login request
    response = requests.post(f"{serviceUrl}login-token", json={'username': getpass(prompt=p[0]), 'token': getpass(prompt=p[1])})

    if response.status_code == 200:  # Check for successful response
        apiKey = response.json()['data']
        print('\nLogin Successful, API Key Received!')
        headers = {'X-Auth-Token': apiKey}
        return apiKey
    else:
        print("\nLogin was unsuccessful, please try again or create an account at: https://ers.cr.usgs.gov/register.")

In [9]:
# login: xdenis88x
# token: CibwxoLdsyu4zAG0NigpSKbS@GX!@K9EjZ8Yv9qXZfiMA7!KcHxbjgftrqLcTdew
serviceUrl = "https://m2m.cr.usgs.gov/api/api/json/stable/"
apiKey = prompt_ERS_login(serviceUrl)

Logging in...


Login Successful, API Key Received!


In [ ]:
fileType = 'band'
bandNames = {'SR_B2', 'QA_PIXEL','ST_B10'} # Blue, Pixel Quality, Thermal Infrared

In [12]:
####################################################
# Open geojson with region of interest
####################################################

from geojson import Polygon, Feature, FeatureCollection, dump

# Open geojson file with area
# Prepare geojson file for downloading
region_geojson = 'data/landsat/regions/yamal.geojson'
aoi_geodf = gpd.read_file(region_geojson)
aoi_geodf = aoi_geodf.to_crs('EPSG:4326')
bounds = aoi_geodf.bounds
bbox = (float(bounds.minx), float(bounds.miny), float(bounds.maxx), float(bounds.maxy))
#with open('./utils/Anchorage_Alaska_aoi.geojson', 'w') as f:
#    dump(feature_collection, f)

# Print bounding box
bbox

(64.206157697069, 62.26622710104113, 83.77694460994564, 73.25371252269805)

In [13]:
import folium
m = folium.Map(location=[aoi_geodf.centroid.y[0], aoi_geodf.centroid.x[0]], zoom_start=8, tiles="openstreetmap",\
              width="90%",height="90%", attributionControl=0) #add n estimate of where the center of the polygon would be located\
                                        #for the location [latitude longitude]

In [14]:
for _, r in aoi_geodf.iterrows():
    sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fillColor": "blue"})
    geo_j.add_to(m)
m


In [15]:
# Define datasaet we want to download
datasetName = 'landsat_ot_c2_l2'

In [16]:
bbox

(64.206157697069, 62.26622710104113, 83.77694460994564, 73.25371252269805)

In [17]:
###################################################
# Define searching parameters
###################################################

# Define spatial filter
spatialFilter = {'filterType' : 'geojson',
                  'geoJson' : {'type': 'Polygon',
                               'coordinates': [[[bbox[0], bbox[1]], [bbox[2], bbox[1]], [bbox[2], bbox[3]], [bbox[0], bbox[3]], [bbox[0], bbox[1]]]]}}

# Define temporal filter
temporalFilter = {'start' : '2024-06-01', 'end' : '2024-06-05'}

# Define cloud parameters
cloudCoverFilter = {'min' : 0, 'max' : 30}

search_payload = {
    'datasetName' : datasetName,
    'sceneFilter' : {
        'spatialFilter' : spatialFilter,
        'acquisitionFilter' : temporalFilter,
        'cloudCoverFilter' : cloudCoverFilter
    }
}

In [18]:
scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)

In [19]:
pd.json_normalize(scenes['results'])

,browse,cloudCover,entityId,displayId,orderingId,metadata,hasCustomizedMetadata,publishDate,options.bulk,options.download,...,options.secondary,selected.bulk,selected.compare,selected.order,spatialBounds.type,spatialBounds.coordinates,spatialCoverage.type,spatialCoverage.coordinates,temporalCoverage.endDate,temporalCoverage.startDate
0,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",26,LC81610142024157LGN00,LC08_L2SP_161014_20240605_20240627_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-26 22:30:07-05,True,True,...,False,False,False,False,Polygon,"[[[68.58004, 64.44386], [68.58004, 66.71169], ...",Polygon,"[[[68.58004, 65.0679], [72.35287, 64.44386], [...",2024-06-05 00:00:00,2024-06-05 00:00:00
1,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",13,LC91690122024157LGN00,LC09_L2SP_169012_20240605_20240611_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-11 18:26:28-05,True,True,...,False,False,False,False,Polygon,"[[[58.60914, 67.12149], [58.60914, 69.42025], ...",Polygon,"[[[58.60914, 67.79871], [62.75225, 67.12149], ...",2024-06-05 00:00:00,2024-06-05 00:00:00
2,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",26,LC81700112024156LGN00,LC08_L2SP_170011_20240604_20240627_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-26 19:40:53-05,True,True,...,False,False,False,False,Polygon,"[[[58.68388, 68.43639], [58.68388, 70.76004], ...",Polygon,"[[[58.68388, 69.15535], [63.02602, 68.43639], ...",2024-06-04 00:00:00,2024-06-04 00:00:00
3,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",1,LC81630142024155LGN00,LC08_L2SP_163014_20240603_20240611_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-11 16:14:00-05,True,True,...,False,False,False,False,Polygon,"[[[65.48857, 64.44419], [65.48857, 66.71208], ...",Polygon,"[[[65.48857, 65.06825], [69.26142, 64.44419], ...",2024-06-03 00:00:00,2024-06-03 00:00:00
4,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",10,LC81630152024155LGN00,LC08_L2SP_163015_20240603_20240611_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-11 16:12:17-05,True,True,...,False,False,False,False,Polygon,"[[[64.3934, 63.08988], [64.3934, 65.34203], [6...",Polygon,"[[[64.3934, 63.68903], [68.00531, 63.08988], [...",2024-06-03 00:00:00,2024-06-03 00:00:00
5,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",17,LC91550172024155LGN00,LC09_L2SP_155017_20240603_20240604_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-04 10:39:17-05,True,True,...,False,False,False,False,Polygon,"[[[74.72835, 60.35802], [74.72835, 62.57815], ...",Polygon,"[[[74.72835, 60.90955], [78.05868, 60.35802], ...",2024-06-03 00:00:00,2024-06-03 00:00:00
6,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",20,LC91640112024154LGN00,LC09_L2SP_164011_20240602_20240603_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-03 08:02:22-05,True,True,...,False,False,False,False,Polygon,"[[[67.80509, 68.43848], [67.80509, 70.75734], ...",Polygon,"[[[67.80509, 69.15097], [72.15473, 68.43848], ...",2024-06-02 00:00:00,2024-06-02 00:00:00
7,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",22,LC91640142024154LGN00,LC09_L2SP_164014_20240602_20240603_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-03 08:08:24-05,True,True,...,False,False,False,False,Polygon,"[[[63.82795, 64.44627], [63.82795, 66.70937], ...",Polygon,"[[[63.82795, 65.06451], [67.60538, 64.44627], ...",2024-06-02 00:00:00,2024-06-02 00:00:00
8,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",10,LC91640152024154LGN00,LC09_L2SP_164015_20240602_20240603_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2024-06-03 07:58:24-05,True,True,...,False,False,False,False,Polygon,"[[[62.74104, 63.09243], [62.74104, 65.33979], ...",Polygon,"[[[62.74104, 63.68591], [66.35679, 63.09243], ...",2024-06-02 00:00:00,2024-06-02 00:00:00
9,"[{'id': '5fb4ba12d7ec307f', 'browse

In [20]:
# Print granule names

idField = 'entityId'

entityIds = []

for result in scenes['results']:
     # Add this scene to the list I would like to download if bulk is available
    if result['options']['bulk'] == True:
        entityIds.append(result[idField])
    
entityIds

['LC81610142024157LGN00',
 'LC91690122024157LGN00',
 'LC81700112024156LGN00',
 'LC81630142024155LGN00',
 'LC81630152024155LGN00',
 'LC91550172024155LGN00',
 'LC91640112024154LGN00',
 'LC91640142024154LGN00',
 'LC91640152024154LGN00',
 'LC91640162024154LGN00',
 'LC81650152024153LGN00',
 'LC81650162024153LGN00']

In [21]:
datasetName

'landsat_ot_c2_l2'

In [22]:
listId = f"temp_{datasetName}_list" # customized list id
scn_list_add_payload = {
    "listId": listId,
    'idField' : idField,
    "entityIds": entityIds,
    "datasetName": datasetName
}
scn_list_add_payload

{'listId': 'temp_landsat_ot_c2_l2_list',
 'idField': 'entityId',
 'entityIds': ['LC81610142024157LGN00',
  'LC91690122024157LGN00',
  'LC81700112024156LGN00',
  'LC81630142024155LGN00',
  'LC81630152024155LGN00',
  'LC91550172024155LGN00',
  'LC91640112024154LGN00',
  'LC91640142024154LGN00',
  'LC91640152024154LGN00',
  'LC91640162024154LGN00',
  'LC81650152024153LGN00',
  'LC81650162024153LGN00'],
 'datasetName': 'landsat_ot_c2_l2'}

In [23]:
count = sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey) 
count

653

In [24]:
sendRequest(serviceUrl + "scene-list-get", {'listId' : scn_list_add_payload['entityIds']}, apiKey) 

In [25]:
download_opt_payload = {
    "listId": listId,
    "datasetName": datasetName
}

if fileType == 'band_group':
    download_opt_payload['includeSecondaryFileGroups'] = True

download_opt_payload

{'listId': 'temp_landsat_ot_c2_l2_list', 'datasetName': 'landsat_ot_c2_l2'}

In [26]:
products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
pd.json_normalize(products)

,id,downloadName,displayId,entityId,datasetId,available,filesize,productName,productCode,bulkAvailable,downloadSystem,secondaryDownloads,fileGroups
0,5e83d14fec7cae84,None,LC08_L2SP_001007_20240924_20240928_02_T1,LC80010072024268LGN00,5e83d14f2fc39685,True,851117694,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
1,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_001007_20240924_20240928_02_T1,LC80010072024268LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_001007_20240924_20240928_02_T1,LC80010072024268LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
3,632210d4770592cf,None,LC08_L2SP_001007_20240924_20240928_02_T1,LC80010072024268LGN00,5e83d14f2fc39685,False,851117694,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
4,5e83d14fec7cae84,None,LC08_L2SP_001010_20240924_20240928_02_T2,LC80010102024268LGN00,5e83d14f2fc39685,True,787506218,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2607,632210d4770592cf,None,LC09_L2SP_233011_20221006_20230326_02_T2,LC92330112022279LGN01,5e83d14f2fc39685,False,814568254,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2608,5e83d14fec7cae84,None,LC09_L2SP_233011_20240925_20241001_02_T2,LC92330112024269LGN00,5e83d14f2fc39685,True,702974919,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2609,6448198cc7b442a4,C2L2 Tile Product Files,LC09_L2SP_233011_20240925_20241001_02_T2,LC92330112024269LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2610,6448198c62023764,C2L2 Tile Product Files,LC09_L2SP_233011_20240925_20241001_02_T2,LC92330112024269LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None


In [27]:
filegroups = sendRequest(serviceUrl + "dataset-file-groups", {'datasetName' : datasetName}, apiKey)  
pd.json_normalize(filegroups['secondary'])

,5f85f041c828327a.ls_c2l2_sr_band.id,5f85f041c828327a.ls_c2l2_sr_band.color,5f85f041c828327a.ls_c2l2_sr_band.description,5f85f041c828327a.ls_c2l2_sr_band.displayOrder,5f85f041c828327a.ls_c2l2_sr_band.icon,5f85f041c828327a.ls_c2l2_sr_band.label,5f85f041c828327a.ls_c2l2_sr_band.files,5f85f041c828327a.ls_c2l2_st_band.id,5f85f041c828327a.ls_c2l2_st_band.color,5f85f041c828327a.ls_c2l2_st_band.description,5f85f041c828327a.ls_c2l2_st_band.displayOrder,5f85f041c828327a.ls_c2l2_st_band.icon,5f85f041c828327a.ls_c2l2_st_band.label,5f85f041c828327a.ls_c2l2_st_band.files
0,ls_c2l2_sr_band,None,Landsat Collection-2 Level-2 Surface Reflectan...,1,None,Level-2 Surface Reflectance Bands,"[{'name': 'ANG.txt', 'productIds': {'6453e8a2f...",ls_c2l2_st_band,None,Landsat Collection-2 Level-2 Surface Temperatu...,1,None,Level-2 Surface Temperature Bands,"[{'name': 'ANG.txt', 'productIds': {'6453e8a2f..."


In [28]:
fileGroupIds = {"ls_c2l2_sr_band"}

In [29]:
# Select products
print("Selecting products...")
downloads = []
if fileType == 'bundle':
    # Select bundle files
    print("    Selecting bundle files...")
    for product in products:        
        if product["bulkAvailable"] and product['downloadSystem'] != 'folder':               
            downloads.append({"entityId":product["entityId"], "productId":product["id"]})


elif fileType == 'band':
    # Select band files
    print("    Selecting band files...")
    for product in products:  
        if product["secondaryDownloads"] is not None and len(product["secondaryDownloads"]) > 0:
            for secondaryDownload in product["secondaryDownloads"]:
                for bandName in bandNames:
                    if secondaryDownload["bulkAvailable"] and bandName in secondaryDownload['displayId']:
                        downloads.append({"entityId":secondaryDownload["entityId"], "productId":secondaryDownload["id"]})


elif fileType == 'band_group':        
    # Get secondary dataset ID and file group IDs with the scenes
    print("    Checking for scene band groups and get secondary dataset ID and file group IDs with the scenes...")
    sceneFileGroups = []
    entityIds = []
    datasetId = None
    for product in products:  
        if product["secondaryDownloads"] is not None and len(product["secondaryDownloads"]) > 0:
            for secondaryDownload in product["secondaryDownloads"]:
                if secondaryDownload["bulkAvailable"] and secondaryDownload["fileGroups"] is not None:
                    if datasetId == None:
                        datasetId = secondaryDownload['datasetId']
                    for fg in secondaryDownload["fileGroups"]:                            
                        if fg not in sceneFileGroups:
                            sceneFileGroups.append(fg)
                        if secondaryDownload['entityId'] not in entityIds:
                            entityIds.append(secondaryDownload['entityId'])

    # Send dataset request to get the secondary dataset name by the dataset ID
    data_req_payload = {
        "datasetId": datasetId,
    }
    results = sendRequest(serviceUrl + "dataset", data_req_payload, apiKey)
    secondaryDatasetName = results['datasetAlias']

    # Add secondary scenes to a list
    secondaryListId = f"temp_{datasetName}_scecondary_list" # customized list id
    sec_scn_add_payload = {
        "listId": secondaryListId,
        "entityIds": entityIds,
        "datasetName": secondaryDatasetName
    }

    print("    Adding secondary scenes to list...")
    count = sendRequest(serviceUrl + "scene-list-add", sec_scn_add_payload, apiKey)    
    print("    Added", count, "secondary scenes\n")

    # Compare the provided file groups Ids with the scenes' file groups IDs
    if fileGroupIds:
        fileGroups = []
        for fg in fileGroupIds:
            fg = fg.strip() 
            if fg in sceneFileGroups:
                fileGroups.append(fg)
    else:
        fileGroups = sceneFileGroups
else:
    # Select all available files
    for product in products:        
        if product["bulkAvailable"]:
            if product['downloadSystem'] != 'folder':            
                downloads.append({"entityId":product["entityId"], "productId":product["id"]})
            if product["secondaryDownloads"] is not None and len(product["secondaryDownloads"]) > 0:
                for secondaryDownload in product["secondaryDownloads"]:
                    if secondaryDownload["bulkAvailable"]:
                        downloads.append({"entityId":secondaryDownload["entityId"], "productId":secondaryDownload["id"]})

Selecting products...
    Selecting band files...


In [30]:
if fileType != 'band_group':
    download_req2_payload = {
        "downloads": downloads,
        "label": label
    }
else:
    if len(fileGroups) > 0:
        download_req2_payload = {
            "dataGroups": [
                {
                    "fileGroups": fileGroups,
                    "datasetName": secondaryDatasetName,
                    "listId": secondaryListId
                }
            ],
            "label": label
        }
    else:
        print('No file groups found')
        sys.exit()

print(f"Sending download request ...")
download_request_results = sendRequest(serviceUrl + "download-request", download_req2_payload, apiKey)
print(f"Done sending download request") 

if len(download_request_results['newRecords']) == 0 and len(download_request_results['duplicateProducts']) == 0:
    print('No records returned, please update your scenes or scene-search filter')
    sys.exit()

Sending download request ...
Done sending download request


In [31]:
# Attempt the download URLs
for result in download_request_results['availableDownloads']:
    print(f"Get download url: {result['url']}\n" )
    runDownload(threads, result['url'])
    
preparingDownloadCount = len(download_request_results['preparingDownloads'])
preparingDownloadIds = []
if preparingDownloadCount > 0:
    for result in download_request_results['preparingDownloads']:
        preparingDownloadIds.append(result['downloadId'])

    download_ret_payload = {"label" : label}                
    # Retrieve download URLs
    print("Retrieving download urls...\n")
    download_retrieve_results = sendRequest(serviceUrl + "download-retrieve", download_ret_payload, apiKey, False)
    if download_retrieve_results != False:
        print(f"    Retrieved: \n" )
        for result in download_retrieve_results['available']:
            if result['downloadId'] in preparingDownloadIds:
                preparingDownloadIds.remove(result['downloadId'])
                runDownload(threads, result['url'])
                print(f"       {result['url']}\n" )
            
        for result in download_retrieve_results['requested']:   
            if result['downloadId'] in preparingDownloadIds:
                preparingDownloadIds.remove(result['downloadId'])
                runDownload(threads, result['url'])
                print(f"       {result['url']}\n" )
    
    # Didn't get all download URLs, retrieve again after 30 seconds
    while len(preparingDownloadIds) > 0: 
        print(f"{len(preparingDownloadIds)} downloads are not available yet. Waiting for 30s to retrieve again\n")
        time.sleep(30)
        download_retrieve_results = sendRequest(serviceUrl + "download-retrieve", download_ret_payload, apiKey, False)
        if download_retrieve_results != False:
            for result in download_retrieve_results['available']:                            
                if result['downloadId'] in preparingDownloadIds:
                    preparingDownloadIds.remove(result['downloadId'])
                    print(f"    Get download url: {result['url']}\n" )
                    runDownload(threads, result['url'])
                    
print("\nDownloading files... Please do not close the program\n")
for thread in threads:
    thread.join()

Get download url: https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2024/001/007/LC08_L2SP_001007_20240924_20240928_02_T1/LC08_L2SP_001007_20240924_20240928_02_T1_QA_PIXEL.TIF?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6MjczNjI2MzUsImRvd25sb2FkSWQiOjgwMDEwMTIyNywiZGF0ZUdlbmVyYXRlZCI6IjIwMjUtMDUtMTZUMDk6NDU6MDAtMDU6MDAiLCJpZCI6IkxDMDhfTDJTUF8wMDEwMDdfMjAyNDA5MjRfMjAyNDA5MjhfMDJfVDFfUUFfUElYRUwuVElGIiwic2lnbmF0dXJlIjoiJDUkJHIxNnJ4RFl2M3haNUJIODZWY2JOcDl3TE9QYmVSMVpoQTMxVC5YMTI4dzEifQ==

Get download url: https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2024/001/007/LC08_L2SP_001007_20240924_20240928_02_T1/LC08_L2SP_001007_20240924_20240928_02_T1_SR_B2.TIF?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6MjczNjI2MzUsImRvd25sb2FkSWQiOjgwMDEwMTIzMywiZGF0ZUdlbmVyYXRlZCI6IjIwMjUtMDUtMTZUMDk6NDU6MDAtMDU6MDAiLCJpZCI6IkxDMDhfTDJTUF8wMDEwMDdfMjAyNDA5MjRfMjAyNDA5MjhfMDJfVDFfU1JfQjIuVElGIiwic2lnbmF0dXJlIjoiJDUkJDhBMW5uM05Ec